# 03 — Data Preprocessing
**Proyek:** Klasifikasi Tingkat Risiko Stroke Berdasarkan Data Klinis dan Gaya Hidup Menggunakan Algoritma TabNet dengan Interpretasi Attention Mechanism

**Tahap:** Data Preprocessing menuju format siap TabNet

**Tujuan:**
1. Menangani missing value (`bmi`).
2. Encoding fitur kategorikal (TabNet butuh integer categorical).
3. Mitigasi outlier (IQR capping).
4. Standardisasi fitur numerik.
5. Split train / validation / test (stratified).
6. Balancing kelas minoritas menggunakan **SMOTE** (hanya pada train).
7. Menyimpan dataset final + artefak scaler/encoder untuk inference.


## 1. Setup

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
METADATA_DIR = PROJECT_ROOT / "metadata"
for p in [DATA_INTERIM, DATA_PROCESSED, METADATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

with open(PROJECT_ROOT / "params.yaml") as f:
    params = yaml.safe_load(f)

RANDOM_STATE = params["dataset"]["random_state"]
print(json.dumps(params, indent=2))


{
  "dataset": {
    "name": "healthcare-dataset-stroke-data",
    "source": "Kaggle - fedesoriano/stroke-prediction-dataset",
    "target": "stroke",
    "random_state": 42
  },
  "preprocessing": {
    "missing_strategy": {
      "bmi": "median",
      "smoking_status": "keep_unknown"
    },
    "drop_columns": [
      "id"
    ],
    "categorical_features": [
      "gender",
      "ever_married",
      "work_type",
      "Residence_type",
      "smoking_status"
    ],
    "numeric_features": [
      "age",
      "avg_glucose_level",
      "bmi"
    ],
    "binary_features": [
      "hypertension",
      "heart_disease"
    ],
    "outlier": {
      "method": "iqr_cap",
      "whisker": 1.5
    },
    "scaling": "standard",
    "balance": {
      "method": "smote",
      "sampling_strategy": 0.5
    },
    "split": {
      "test_size": 0.15,
      "val_size": 0.15,
      "stratify": true
    }
  },
  "tabnet": {
    "n_d": 16,
    "n_a": 16,
    "n_steps": 5,
    "gamma": 1.5,
    "l

## 2. Load Data Mentah

In [2]:
df = pd.read_csv(DATA_RAW / "healthcare-dataset-stroke-data.csv")
print(df.shape)
df.head()


(5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. Drop Kolom Non-informatif

In [3]:
df = df.drop(columns=params["preprocessing"]["drop_columns"])
print("Kolom tersisa:", df.columns.tolist())


Kolom tersisa: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']


Kolom `id` dihapus karena bersifat identifier dan **tidak memiliki makna prediktif** — justru berpotensi menjadi *noise feature* bagi model.

## 4. Penanganan Missing Value

In [4]:
print("Missing sebelum imputasi:")
print(df.isna().sum())


Missing sebelum imputasi:
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [5]:
# Imputasi bmi dengan median (robust terhadap outlier)
bmi_median = df["bmi"].median()
df["bmi"] = df["bmi"].fillna(bmi_median)
print(f"Imputed bmi median = {bmi_median:.2f}")

# smoking_status: pertahankan kategori 'Unknown' sesuai keputusan di EDA
print("\nMissing setelah imputasi:")
print(df.isna().sum())


Imputed bmi median = 28.10

Missing setelah imputasi:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


## 5. Gender = 'Other' dihapus (hanya 1 baris)

In [6]:
print(df["gender"].value_counts())
df = df[df["gender"].isin(["Male", "Female"])].copy()
print("\nShape setelah filter gender:", df.shape)


gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

Shape setelah filter gender: (5109, 11)


## 6. Outlier Handling (IQR Capping / Winsorizing)

In [7]:
def iqr_cap(series: pd.Series, whisker: float = 1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - whisker * iqr, q3 + whisker * iqr
    return series.clip(lower, upper), (lower, upper)

cap_info = {}
for col in params["preprocessing"]["numeric_features"]:
    df[col], bounds = iqr_cap(df[col], whisker=params["preprocessing"]["outlier"]["whisker"])
    cap_info[col] = {"lower": float(bounds[0]), "upper": float(bounds[1])}
print(json.dumps(cap_info, indent=2))


{
  "age": {
    "lower": -29.0,
    "upper": 115.0
  },
  "avg_glucose_level": {
    "lower": 21.964999999999982,
    "upper": 169.365
  },
  "bmi": {
    "lower": 10.300000000000006,
    "upper": 46.29999999999999
  }
}


Outlier ekstrem di `avg_glucose_level` & `bmi` **di-cap** ke bound IQR, bukan dihapus — supaya baris minoritas (stroke=1) yang sudah sedikit tidak makin hilang.

## 7. Encoding Fitur Kategorikal (Integer Encoding untuk TabNet)

In [8]:
cat_features = params["preprocessing"]["categorical_features"]

encoders = {}
cat_dims = {}
for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = {cls: int(i) for i, cls in enumerate(le.classes_)}
    cat_dims[col] = len(le.classes_)

print("Mapping encoding:")
print(json.dumps(encoders, indent=2))
print("\nDimensi kategori:")
print(cat_dims)
df.head()


Mapping encoding:
{
  "gender": {
    "Female": 0,
    "Male": 1
  },
  "ever_married": {
    "No": 0,
    "Yes": 1
  },
  "work_type": {
    "Govt_job": 0,
    "Never_worked": 1,
    "Private": 2,
    "Self-employed": 3,
    "children": 4
  },
  "Residence_type": {
    "Rural": 0,
    "Urban": 1
  },
  "smoking_status": {
    "Unknown": 0,
    "formerly smoked": 1,
    "never smoked": 2,
    "smokes": 3
  }
}

Dimensi kategori:
{'gender': 2, 'ever_married': 2, 'work_type': 5, 'Residence_type': 2, 'smoking_status': 4}


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1,67.0,0,1,1,2,1,169.365,36.6,1,1
1,0,61.0,0,0,1,3,0,169.365,28.1,2,1
2,1,80.0,0,1,1,2,0,105.920,32.5,2,1
3,0,49.0,0,0,1,2,1,169.365,34.4,3,1
4,0,79.0,1,0,1,3,0,169.365,24.0,2,1


> TabNet **bekerja lebih baik dengan integer-encoded categorical** + parameter `cat_idxs` & `cat_dims` daripada one-hot, karena punya embedding layer internal yang akan belajar representasinya sendiri.

## 8. Pisahkan Fitur (X) dan Target (y)

In [9]:
y = df["stroke"].values
X = df.drop(columns=["stroke"])
feature_names = X.columns.tolist()
print("Fitur:", feature_names)
print("X shape:", X.shape, "| y shape:", y.shape)
print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))


Fitur: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']
X shape: (5109, 10) | y shape: (5109,)
Class distribution: {np.int64(0): np.int64(4860), np.int64(1): np.int64(249)}


## 9. Split Train / Validation / Test (Stratified)

In [10]:
test_size = params["preprocessing"]["split"]["test_size"]
val_size = params["preprocessing"]["split"]["val_size"]

# Step 1: train_val vs test
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE
)
# Step 2: train vs val (proporsi val dari sisa)
val_ratio = val_size / (1 - test_size)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=val_ratio, stratify=y_tv, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape} | positives: {y_train.sum()}")
print(f"Val   : {X_val.shape} | positives: {y_val.sum()}")
print(f"Test  : {X_test.shape} | positives: {y_test.sum()}")


Train : (3575, 10) | positives: 175
Val   : (767, 10) | positives: 37
Test  : (767, 10) | positives: 37


## 10. Standardisasi Fitur Numerik

In [11]:
numeric_features = params["preprocessing"]["numeric_features"]
scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

print("Mean train:", X_train[numeric_features].mean().round(4).tolist())
print("Std  train:", X_train[numeric_features].std().round(4).tolist())


Mean train: [-0.0, -0.0, 0.0]
Std  train: [1.0001, 1.0001, 1.0001]


> Scaler **hanya di-fit pada data train**, kemudian ditransformasikan ke val & test — mencegah *data leakage*.

## 11. Balancing dengan SMOTE (khusus Train)

In [12]:
sampling_strategy = params["preprocessing"]["balance"]["sampling_strategy"]
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Sebelum SMOTE:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Setelah SMOTE:", dict(zip(*np.unique(y_train_bal, return_counts=True))))


Sebelum SMOTE: {np.int64(0): np.int64(3400), np.int64(1): np.int64(175)}
Setelah SMOTE: {np.int64(0): np.int64(3400), np.int64(1): np.int64(1700)}


**Penting:** SMOTE **hanya pada set training**. Val & test dibiarkan apa adanya supaya evaluasi realistis terhadap distribusi asli.

`sampling_strategy=0.5` artinya minoritas di-*oversample* hingga mencapai 50% dari kelas mayoritas (tidak penuh 1:1, supaya tidak over-synthetic).

## 12. Simpan Artefak & Dataset Final

In [13]:
# Dataset final → CSV
X_train_bal.assign(stroke=y_train_bal).to_csv(DATA_PROCESSED / "train.csv", index=False)
X_val.assign(stroke=y_val).to_csv(DATA_PROCESSED / "val.csv", index=False)
X_test.assign(stroke=y_test).to_csv(DATA_PROCESSED / "test.csv", index=False)

# NumPy format (siap TabNet)
np.savez_compressed(
    DATA_PROCESSED / "tabnet_ready.npz",
    X_train=X_train_bal.values.astype(np.float32),
    y_train=y_train_bal.astype(np.int64),
    X_val=X_val.values.astype(np.float32),
    y_val=y_val.astype(np.int64),
    X_test=X_test.values.astype(np.float32),
    y_test=y_test.astype(np.int64),
)

# Scaler + encoders + cat_dims
joblib.dump(scaler, DATA_PROCESSED / "scaler.joblib")
with open(DATA_PROCESSED / "encoders.json", "w") as f:
    json.dump(encoders, f, indent=2)

# TabNet config: cat_idxs & cat_dims
cat_idxs = [feature_names.index(c) for c in cat_features]
tabnet_meta = {
    "feature_names": feature_names,
    "cat_features": cat_features,
    "cat_idxs": cat_idxs,
    "cat_dims": [cat_dims[c] for c in cat_features],
    "numeric_features": numeric_features,
    "class_counts": {
        "train": dict(zip(*[list(map(int, a)) for a in np.unique(y_train_bal, return_counts=True)])),
        "val":   dict(zip(*[list(map(int, a)) for a in np.unique(y_val, return_counts=True)])),
        "test":  dict(zip(*[list(map(int, a)) for a in np.unique(y_test, return_counts=True)])),
    },
}
with open(DATA_PROCESSED / "tabnet_meta.json", "w") as f:
    json.dump(tabnet_meta, f, indent=2)

print("Artefak preprocessing tersimpan di:", DATA_PROCESSED)
for p in sorted(DATA_PROCESSED.iterdir()):
    print(" •", p.name, "-", p.stat().st_size, "bytes")


Artefak preprocessing tersimpan di: /sessions/gracious-peaceful-bohr/mnt/Riset Data Mining/data/processed
 • encoders.json - 395 bytes
 • scaler.joblib - 959 bytes
 • tabnet_meta.json - 694 bytes
 • tabnet_ready.npz - 69471 bytes
 • test.csv - 57708 bytes
 • train.csv - 381698 bytes
 • val.csv - 57648 bytes


## 13. Simpan Metadata Preprocessing

In [14]:
preprocessing_metadata = {
    "steps": [
        "drop_id",
        "impute_bmi_median",
        "remove_gender_other",
        "iqr_cap_outliers",
        "label_encode_categoricals",
        "stratified_split",
        "standard_scale_numeric",
        "smote_train_only",
    ],
    "imputation": {"bmi": float(bmi_median)},
    "iqr_cap": cap_info,
    "encoders": encoders,
    "split_sizes": {
        "train_before_smote": int(len(y_train)),
        "train_after_smote":  int(len(y_train_bal)),
        "val":  int(len(y_val)),
        "test": int(len(y_test)),
    },
    "scaler": {
        "type": "StandardScaler",
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    },
    "random_state": RANDOM_STATE,
}
with open(METADATA_DIR / "03_preprocessing.json", "w") as f:
    json.dump(preprocessing_metadata, f, indent=2)
print("Metadata preprocessing tersimpan ✓")


Metadata preprocessing tersimpan ✓
